# Accuracy assessment — Siddipet (maize) & Osmanabad (Bengal gram)

GP-level validation of predicted yield rasters against CCE points.

**Inputs:** yield rasters, CCE CSVs, GP shapefiles  
**Outputs:** GP metrics workbook  

> Update the path variables in the first cells before running. See [`docs/`](../../docs/) for methodology and parameters.

In [ ]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import rasterio as rio

In [ ]:
from scipy.stats import pearsonr

In [ ]:
import geopandas as gpd

In [ ]:
import pandas as pd

In [ ]:
from shapely import Point
import numpy as np

In [ ]:
import os

In [ ]:
out_path=r'C:\Users\PushkarGaur\Downloads\output'
if not os.path.exists(out_path):
    os.mkdir(out_path)

In [ ]:
def get_coord(shape):
    shapefile=pd.read_csv(shape)
    coords = [(x,y) for x, y in zip(shapefile.longitude, shapefile.latitude)]
    return coords

In [ ]:
def getRasterValue(image,coords):
    ras = rio.open(image)
    return [x[0] for x in ras.sample(coords)]

In [ ]:
def mape(y_true, y_pred): 
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100


In [ ]:
def accuracy_parameter(df):
    y_test=np.array(df['Yield(kg_p_ha)'])*0.87
    y_pred=np.array(df['Predicted'])*0.87
    mae = mean_absolute_error(y_true=y_test,y_pred=y_pred)
    mse = mean_squared_error(y_true=y_test,y_pred=y_pred) #default=True
    rmse = mean_squared_error(y_true=y_test,y_pred=y_pred,squared=False)
    if len(y_test)>2:
        r2 = pearsonr(y_test,y_pred)[0]
    else:
        r2=0
    maper = mape(y_test,y_pred)
    test_mean=y_test.mean()
    pred_mean=y_pred.mean()
    # print("MAE:",round(mae,2))
    # print("MSE:",round(mse,2))
    # print("RMSE:",round(rmse,2))
    # print("R2:",round(r2,2))
    # print("MAPE:",round(maper,2))
    
    return mae,mse,rmse,r2,maper,test_mean, pred_mean

In [ ]:
raster = r'C:\Users\PushkarGaur\Downloads\Osmanabad_bengal_gram_yield\Osmanabad_bengal_gram_yield.tif'
shapefile = r'C:\Users\PushkarGaur\Downloads\Siddipet GP\Siddipet.shp'
points = r'C:\Users\PushkarGaur\Downloads\Siddipet_Maize_Rabi2022-23\siddipet.csv'

In [ ]:
shp = gpd.read_file(shapefile)
pts = pd.read_csv(points)

In [ ]:
geometry = [Point(xy) for xy in zip(pts['longitude'], pts['latitude'])]
pt_gdf = gpd.GeoDataFrame(pts, crs='EPSG:4326', geometry=geometry)
shp = shp.to_crs(pt_gdf.crs)
# pt_gdf['Predicted']=getRasterValue(raster,get_coord(points))
# pt_gdf=pt_gdf[pt_gdf['Predicted']>0]
# pt_gdf['difference']=abs(pt_gdf['Predicted']-pt_gdf['Yield(kg_p_ha)'])
# pt_gdf=pt_gdf[pt_gdf['difference']<500]

In [ ]:
import seaborn as sns

In [ ]:
test_df = gpd.sjoin(pt_gdf,shp,how='inner', predicate='intersects')

In [ ]:
shp.columns

In [ ]:
grp = test_df.groupby('GP_Name')
data=[]
npts=[]
fid=[]
for d in grp:
    npts.append(d[1].shape[0])
    data.append(accuracy_parameter(d[1]))
    fid.append(d[0])

In [ ]:
df=pd.DataFrame(data,columns=['MAE','MSE','RMSE','R','MAPE','Observed','Predicted'])

In [ ]:
df['GP Name']=fid

In [ ]:
df=df[df['MAPE']<50]

In [ ]:
df=df.drop(columns=['R','MSE'])

In [ ]:
df.to_excel(os.path.join(out_path,'Siddipet_gp_mc.xlsx'))

In [ ]:
df